<a href="https://colab.research.google.com/github/nagmafarez/IC-Assignments/blob/main/RAG_ChatBot_HealthCare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **What is Basic RAG?**
Basic RAG is the standard,straightforward implementation of Retrieval-Augmented Generation. It involves relevant information from a knowledge base in response to a query, then using this information to generate an answer using a language model.

# ▶ **Why we need RAG?**

1. Combines the broad knowledge of language models with specific, up-to-date information.
2. Improves accuracy of responses by grounding them in retrieved facts.
3. Reduces hallucinations common in standalone language models.
4. Allows for easy updates to the knowledge base without retraining the entire model.

Step 1 : Library Installation

In [ ]:
# Installing Necessary Libraries
!pip install langchain langchain-groq langchain-community sentence-transformers faiss-gpu-cu12 pypdf langchain-huggingface

Step 2 : Import Modules & Set API Keys

In [ ]:
# Import Modules
import os
from getpass import getpass

In [ ]:
# Setting API Keys
os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")
os.environ["HF_TOKEN"] = getpass("Enter HuggingFace Token: ")

Enter Groq API Key: ··········
Enter HuggingFace Token: ··········


In [ ]:
!pip install langchain_classic

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import faiss
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

Step 3: Load and Process the documents.

We are using pyPDFLoader to load the PDF and then split it into smaller chunks so the LLM can process it effectively.

In [ ]:
# Load the PDF
pdf_path = "/content/health care.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

In [ ]:
# Splitting text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 529


In [ ]:
# Initializing Embeddings
embedding = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-m3",
    model_kwargs = {'trust_remote_code': True}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
# Creating Vector store
vectorstore = faiss.FAISS.from_documents(chunks, embedding)
print("Vector Store Created Successfully!")

Vector Store Created Successfully!


Step 5 : Intialize the LLM Groq

We use Groq's API to access open-source models like Mixtral or Llama3 or Gemma with incrcedible speed.

In [ ]:
llm = ChatGroq(
    temperature=0,
    model_name = "llama-3.3-70b-versatile"
)

Step 6: Create the Retrieval Chain

Finally, we connect the retriever (vector store) and the LLM using a RetrievalQA chain.

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

Step 7: Testing the RAG Chatbot

In [ ]:
# Defining Promt Template
# RetrievalQA expects exactly '{context}' and '{question}' as variables
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Helpful Answer:"""

In [ ]:
from langchain_classic.prompts import PromptTemplate

In [ ]:
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

In [ ]:
# Initializing RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # "stuffing" all context into the prompt
    retriever=vectorstore.as_retriever(),
    return_source_documents=True, # Useful to see which PDF pages were used
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT} # Passing the custom prompt
)

In [ ]:
# Run the Query
question = "What is the main topic of this document?"
response = qa_chain.invoke({"query": question})

In [ ]:
# Print Result
print("Answer:", response["result"])

Answer: The main topic of this document appears to be related to health and behavioral responses to chronic health conditions, specifically how people change their behaviors after being diagnosed with certain diseases, such as high blood pressure, diabetes, and cancer.


In [37]:
from google.colab import output
output.disable_custom_widget_manager()